# Parallel Batch Motion Correction Demo (TIFF + Binning)

This notebook demonstrates how to process `.tif` microscopy files using a parallelized bin-register-interpolate-apply workflow based on Pystackreg.

Requires installation of: pystackreg, zarr and joblib

- Loads `.tif` files (TIFF stacks)
- Temporally bins the data for robust registration
- Calculates motion correction transforms on binned data
- Interpolates transforms to full frame rate
- Applies transforms to original data
- Saves output as `.tif` 

In [1]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import os
import sys

sys.path.append('../../src')

from parallel_motion_correct import (
    parallel_batch_process,
    find_files,
    get_available_cores,
    load_data
)

print(f"Available CPU cores: {get_available_cores()}")

Available CPU cores: 24


## 1. Setup Data Paths
Configure where to find the raw files.

In [ ]:
# Option 1: Load from CSV (User legacy workflow)
file_list = []

try:
    csv_path = '../../RGECO.xlsx'
    if os.path.exists(csv_path):
        allData = pd.read_excel(csv_path)
        # Note: Adjust 'drive' and path mapping if running on Linux with Windows paths in CSV
        drive = '/media/marcotti-nas/' 
        allData['Folder'] = allData['Folder'].str.replace('Z:',drive, regex=False)

        #IF the is is linux or mac
        if sys.platform in ['linux', 'darwin']:  # darwin is macOS

            allData['Folder'] = allData['Folder'].str.replace('\\\\', '/', regex=False)

            allData['Folder'] = allData['Folder'].str.replace('\\', '/', regex=False)

        fileName = '1-jumpCorrected.tif'
        
        # Collect potential paths
        for f in allData['Folder'].values:
            if isinstance(f, str):
                # If paths in CSV are absolute Windows paths, you might need to fix them manually here
                # Here we assume they might be relative or correctable
                path = Path(f) / 'processedMovies' / fileName
                if path.exists():
                    file_list.append(path)
                    
        print(f"Found {len(file_list)} existing files from CSV")
    else:
        print(f"CSV {csv_path} not found, skipping csv loading.")
except Exception as e:
    print(f"Error loading from CSV: {e}")

# Option 2: Search directory (Fallback) - Now searches for .tif files
if not file_list:
    search_dir = Path('.') # Current directory or replace with data path
    print(f"Searching {search_dir.resolve()} for .tif files...")
    # Search for TIFF files instead of RAW files
    file_list = list(search_dir.rglob('*.tif')) + list(search_dir.rglob('*.tiff'))
    print(f"Found {len(file_list)} TIFF files in directory search")
    
# Display found files
for f in file_list[:5]:
    print(f"  - {f}")
if len(file_list) > 5:
    print(f"  ... and {len(file_list)-5} more")

Found 1 existing files from CSV
  - /media/marcotti-lab/Users/Char/2PI Data/2025/2025_11_04_TMEM16A-Pax2Cre-GCaMP6f_PHPebMyo15CreRGECO AAV_in vivo_P8_Tattoo 1/1_001/processedMovies/1-jumpCorrected.tif


## 2. Configure Processing Parameters

In [ ]:
config = {
    'output_dir':None, # Base output dir (Note: actual files saved in 'corrected' subfolder of input)
    'transform_type': 'rigid',     # 'translation', 'rigid', 'scaled', 'affine', 'bilinear'
    'reference_frames': 100,        # Frames to average for reference image
    'temporal_bin': 3,            # Frames to average together for robust registration
    'n_frames': None,              # Process all frames (None)
    'save_transforms': False,       # Save transformation matrices as .npy
    'n_jobs': 15,                  # Use all available cores
    'apply_to_full': True,          # False: save the binned, True: save the original file using the interpolated transformation from the binned file. 
    'save_binned' : False,           # Save also the binned file if apply_to_full is True
    'chunk_size' : 3000,
    'skip_existing': False,      # ← Add this
    'suffix' : '-mc',
    'correct_intensity': True,    # Apply intensity correction to avoid brightness shifs
    'companion_suffix':None,
    'register_on_companion':False,  # ← Registration source is now the companion,
    'backend': 'loky'             # Joblib backend: 'loky' (default), 'threading', or 'multiprocessing'. Try 'threading' if having issues on Windows.
}

print("Configuration:")
for k, v in config.items():
    print(f"  {k}: {v}")

Configuration:
  output_dir: None
  transform_type: rigid
  reference_frames: 100
  temporal_bin: 1
  n_frames: None
  save_transforms: False
  n_jobs: 15
  apply_to_full: True
  save_binned: False
  chunk_size: 3000
  skip_existing: False
  suffix: -mc
  correct_intensity: True
  companion_suffix: -channel2
  register_on_companion: True
  backend: loky


## 3. Run Parallel Processing

In [5]:
if file_list:
    results = parallel_batch_process(
        file_list=file_list,
        **config
    )
else:
    print("No files to process!")
    results = []

2026-02-11 11:02:28,606 - INFO - Using memory-efficient chunked processing (chunk_size=3000)
2026-02-11 11:02:28,609 - INFO - Processing 1 files with 15 parallel jobs (backend=loky)
[Parallel(n_jobs=15)]: Using backend LokyBackend with 15 concurrent workers.
2026-02-11 11:02:30,246 - INFO - Processing 1-jumpCorrected.tif: 6446 frames @ 416x1024
2026-02-11 11:02:30,246 - INFO - Computing transforms on companion file: 1-jumpCorrected-channel2.tif
2026-02-11 11:02:30,246 - WARNING - No temporal binning - loading full stack for registration
2026-02-11 11:02:50,618 - INFO - Phase 1: Computing transforms on 6446 binned frames...
2026-02-11 11:02:53,257 - INFO - Brightest frame: 2269 (mean intensity: 143.1). Using reference frames 2219-2318
Motion correction: 100%|██████████| 6446/6446 [1:43:50<00:00,  1.03frame/s]
2026-02-11 12:52:04,920 - INFO - Will apply intensity correction: offset = -2.25
2026-02-11 12:53:32,258 - INFO - Phase 3: Applying transforms to 1 additional file(s)...
2026-02-11

## 4. Summary

In [5]:
print(f"{'='*80}")
print("PROCESSING SUMMARY")
print(f"{'='*80}")
print(f"{'File':<40} {'Status':<10} {'Frames':<10} {'Time (s)':<10}")
print(f"{'-'*80}")

successful = []
failed = []

for r in results:
    status = "✓" if r.success else "✗"
    name = r.input_path.name[:38]
    print(f"{name:<40} {status:<10} {r.n_frames:<10} {r.processing_time:<10.1f}")
    if r.success:
        successful.append(r)
    else:
        failed.append(r)

print(f"{'-'*80}")
print(f"Total: {len(successful)} succeeded, {len(failed)} failed")

if failed:
    print("\nErrors:")
    for r in failed:
        print(f"{r.input_path}: {r.error_message}")

PROCESSING SUMMARY
File                                     Status     Frames     Time (s)  
--------------------------------------------------------------------------------
1-jumpCorrected.tif                      ✓          6965       3570.0    
--------------------------------------------------------------------------------
Total: 1 succeeded, 0 failed
